Conda environment: txtai

In [1]:
from txtai import Embeddings, LLM
from txtai.pipeline import Textractor
from pathlib import Path

textractor = Textractor(chunker="late", paragraphs=True)
embeddings = Embeddings({"path": "ollama/qwen3-embedding:4b", "content":True, "hybrid":  True, "db": "sqlite://sample.db"})
embeddings_graph = Embeddings({
  "autoid": "uuid5",
  "path": "intfloat/e5-base",
  "instructions": {
    "query": "query: ",
    "data": "passage: "
  },
  "content": True,
  "hybrid": True,
  "graph": {
      "approximate": False,
      "topics": {}
  }
})
llm = LLM(path="ollama/qwen3.5:9b")

/home/zbta138a/miniconda3/envs/txtai/lib/python3.14/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/zbta138a/miniconda3/envs/txtai/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 134/134 [00:00<00:00, 4431.56it/s]
/home/zbta138a/miniconda3/envs/txtai/lib/python3.14/site-packages/chonkie/embed

In [2]:

import uuid
markdown = Path("outputs/energies-18-00645/auto/energies-18-00645.md").read_text(encoding="utf-8")
# Extract headings and filter out empty ones
text_from_headings = markdown.split("##")
headers = [h.split('\n')[0].strip() for h in markdown.split("##") if h.strip()]
documents = []
for p_idx, para in enumerate(text_from_headings):
    postString = para.split("\n",1)[1]
    postString = postString.split("\n\n")

    for text in postString: 
    #print(postString) 
        extracted_text = textractor(text)
        doc_id = str(uuid.uuid4())
        documents.append({"id": doc_id, "text": extracted_text, "heading": headers[p_idx]})


#

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [3]:
documents[10]

{'id': '6046b44e-3ce7-42aa-9b12-02b2f3de5ba1',
 'text': ['Citation: Verheyen, J.; Thommessen, C.; Roes, J.; Hoster, H. Effects on the Unit Commitment of a District Heating System Due to Seasonal Aquifer Thermal Energy Storage and Solar Thermal Integration. Energies 2025, 18, 645. https://doi.org/ 10.3390/en18030645'],
 'heading': 'Article'}

In [4]:
embeddings.index(documents)
embeddings_graph.index(documents)

In [78]:
embeddings.save("scientific_index")

In [5]:
results = embeddings.search("Average air temperature in 2023")
#embeddings.search("What was the abstract of this paper?")
results[0]

{'id': '8c6f46e4-58ec-4a54-a2e3-e1779d191077',
 'text': 'he average air temperature in 2023 is 1The average air temperature in 2023 is $1 1 . 5 ^ { \\circ } \\mathrm { C }$ nd the average river temperature is and the average river temperature is $1 3 . 1 ^ { \\circ } \\mathrm { C }$ The river temperature follows the air temperature except slightly shifted and with . The river temperature follows the air temperature except slightly shifted and with flattened peaks. On the basis of these data, the losses of short-term TES, the COP of the river water HP, or the solar yield of the ST plant are affected.',
 'score': 0.37553107738494873}

In [6]:
# 2) Find in the txtai index (SQL query)
doc_id = results[0]["id"]

indexed = embeddings.search(
    "select id, text, heading from txtai where id = :id",
    limit=1,
    parameters={"id": doc_id},
)
print("Indexed:", indexed)

Indexed: [{'id': '8c6f46e4-58ec-4a54-a2e3-e1779d191077', 'text': 'he average air temperature in 2023 is 1The average air temperature in 2023 is $1 1 . 5 ^ { \\circ } \\mathrm { C }$ nd the average river temperature is and the average river temperature is $1 3 . 1 ^ { \\circ } \\mathrm { C }$ The river temperature follows the air temperature except slightly shifted and with . The river temperature follows the air temperature except slightly shifted and with flattened peaks. On the basis of these data, the losses of short-term TES, the COP of the river water HP, or the solar yield of the ST plant are affected.', 'heading': '2.3. Application Case and Assumptions'}]


In [7]:
def execute(question, text):
  print("--------Contexts --------" )
  print(text)
  print("-----------")
  result=  llm([
    {"role": "system", "content": "You are a scientific assistant. You answer questions from users."},
    {"role": "user", "content": f"""
        Answer the following question using only the context below. Only include information specifically discussed. And make sure you give a source heading as well.

        question: {question}
        context: {text} 
    """}
  ], maxlength=4096)

  print(result)

  return result


In [8]:
import time
def context(question):
  context =  "\n".join(x["text"] for x in embeddings.search(question))
  return context

def rag(question):
  return execute(question, context(question))

start = time.time()
result = rag("Summarize the conclusion of this paper")
print(result)
# print("Took ", end-start, "seconds")
# start = time.time()
result = rag("Tell me about the setup used in this paper")
# print(result)
# start = time.time(
result = rag("Give me the nominal loads, energy sources, and units of controllable DHS plants in a table")
print(result)
# end = time.time()
# print("Took ", end-start, "seconds")


--------Contexts --------
Conflicts of Interest: The authors declare no conflicts of interest. The funders had no role in the design of the study; in the collection, analyses, or interpretation of data; in the writing of the manuscript; or in the decision to publish the results.
Author Contributions: Conceptualization, J.V.; methodology, J.V. and C.T.; software, J.V.; validation, J.V. and C.T.; formal analysis, J.V. and C.T.; investigation, $\mathrm { J . V . } ;$ resources, J.V. and C.T.; data curation, C.T. and J.V.; writing—original draft preparation, J.V. and C.T.; writing—review and editing, J.V., ${ \mathrm { C . T . , J . R } }$ . and H.H.; visualization, J.V. and C.T.; supervision, H.H. and J.R.; project administration, J.R. and ${ \mathrm { C . T . } } ;$ funding acquisition, ${ \mathrm { C . T } } . ,$ J.R. and H.H. All authors have read and agreed to the published version of the manuscript.
Academic Editors: Francesco Calise, Qiuwang Wang, Maria Vicidomini, Wenxiao Chu and P

IndexError: list index out of range

# Trying with RAG module

In [7]:
from txtai import RAG

rag = RAG(
    embeddings_graph,
    context=5,
    path="ollama/gemma4:latest",
    template="""
  
  You are a scientific assistant.
  
  Answer the following question using the provided context. If you cannot find the answer within the context, just say you do not know. 

  Question:
  {question}

  Context:
  {context}
  """,
    output="reference",
)


# user_questions = [
#     "Tell me about the setup used in this paper?",
#     "What was the average air and water temperature in 2023?",
#     "Summarize the Conclusions section of this paper.",
#     "Was AI used?",
# ]

# for question in user_questions:
#     start_time = time.time()
#     response = rag(question, maxlength=4096)
#     end_time = time.time()
#     print("Question:", question)
#     print("Answer:", response)
#     print("Took", end_time - start_time, "seconds")
#     #print("CITATION:", embeddings.search("select id, text from txtai where id = :id", limit=1, parameters={"id": response["reference"]}))


In [8]:
response = rag( "What was the average air and water temperature in 2023?", maxlength=4096)
response

{'answer': 'The average air temperature in 2023 was $11.5^\\circ\\mathrm{C}$, and the average river temperature was $13.1^\\circ\\mathrm{C}$.',
 'reference': '8c6f46e4-58ec-4a54-a2e3-e1779d191077'}

In [9]:
response = rag( "What data did you ingest in your RAG system?", maxlength=4096)
response    

{'answer': 'I do not know.',
 'reference': 'a4d87e7f-999f-4dca-9ff5-6814ef5c1055'}

In [11]:
response = rag( "Give a general overview of this paper?", maxlength=4096)
response

{'answer': 'This paper presents a study centered on an MILP (Mixed-Integer Linear Programming) problem definition concerning DHS heat supply from controllable plants. The model incorporates various elements such as:\n\n*   **Variables:** Includes summary variables for the DHS heat supply, binary operating variables, TES charging or discharging, the TES storage level, and the dimensioning of ST and ATES.\n*   **Assumptions:** Parameters are supported by assumptions summarized in Table 2, which includes factors like solar irradiation and temperature differences used to determine evaporation and condensation temperatures for the ATES-HP.\n*   **Configurations Studied:** The paper discusses specific configurations, including detailed results for scenario A, as well as brief presentations of other ATES configurations (Scenario B: ST with HIT-ATES and HP).\n\nThe document also notes several limitations regarding its findings, specifically that:\n\n1.  The results are highly dependent on the 

In [12]:
response = rag( "Give all mentions of temperature in this paper. What types of temperature were they?", maxlength=4096)
response

{'answer': 'The mentions of temperature found in the context are as follows:\n\n*   **Temperature difference** (of the heat exchangers)\n*   **Evaporation temperature** (of the ATES-HP)\n*   **Condensation temperature** (of the ATES-HP)\n*   **DHS supply temperature** ($105^\\circ\\mathrm{C}$)\n*   **Ambient air temperatures** (in Berlin)\n*   **River water temperatures** (of the Spree)\n*   **Injection well temperature** ($20^\\circ\\mathrm{C}$)',
 'reference': '0c503b4c-6445-446a-867b-2743511feae2'}

In [13]:
response = rag( "What setup was used in the experiments?", maxlength=4096)
response

{'answer': 'The context describes several different setups and models used in various studies:\n\n*   **ATES-HP:** The calculation uses solar irradiation or temperature difference of heat exchangers to determine the evaporation and condensation temperature for the ATES-HP.\n*   **Single House/Small Scale:** Miglani et al. developed a framework optimizing ground source HPs and ST for a single house, considering solar photovoltaic, EHBs, and small-scale TES.\n*   **District Level (52 buildings):** Renaldi and Friedrich conducted a multi-year parametric study for the technoeconomic design of ST and BTES without HPs on a district level including 52 buildings.\n*   **50-House Development:** Reed et al. performed a financial model calculation for a 50-house development with DHS, incorporating BTES and ST compared to individual heating using natural gas-fired heat-only boilers (HOBs).',
 'reference': 'b6d5a2d2-68b4-47b0-bad6-b610c2d47403'}

In [14]:
response = rag( "Give me the nominal loads, energy sources, and units of controllable DHS plants in a table.", maxlength=4096)
response

{'answer': 'Based on Table 1, here is the table showing the nominal loads, energy sources, and related details for the controllable DHS plants:\n\n| Index | Plant Type | Energy Source | Electricity Nominal Load ($P_{max}$) (MW) | Heat Supply Nominal Load ($\\dot{Q}_{max}$) (MW) | Fuel Consumption Nominal Load ($\\dot{Q}_f$) (MW) |\n| :---: | :----------: | :-----------: | :----------------------: | :-----------------------------: | :-------------------------: |\n| CHP 1 | TPS1 | Biomass | 20 | 30 | 5 |\n| CHP 2 | Engine9 | Natural gas | 2.9 | 3 | 3.08 |\n| CHP 3 | Engine1 | Natural gas | 0.4 | 2 | 0.5 |\n| CHP 4 | Engine1 | Natural gas | 0.8 | - | 0.8 |\n| HOB 1 | HOB2 | Natural gas | 18.5 | 2 | 0.7 |\n| HOB 2 | HOB3 | Natural gas | 3 | 3 | 6.4 |\n| HOB 3 | HOB2 | Natural gas | 20 | 2 | 0.6 |\n| HOB 4 | HOB1 | Natural gas | 10 | 1 | 0.4 |\n| HOB 5 | HOB2 | Natural gas | - | 22.2 | - |\n| HOB 6 | HOB10 | Natural gas | 2.7 | 6 | 3.0 |\n| HOB 7 | HOB1 | Natural gas | 0.9 | 6 | 1.4 |\n| EH

In [18]:
# 2) Find in the txtai index (SQL query)
doc_id = response["reference"]
try:
    indexed = embeddings.search(
        "select id, text, heading from txtai where id = :id",
        limit=1,
        parameters={"id": doc_id},
    )
    print("Indexed:", indexed)
except Exception as e:
    print("Index query error:", e)

Indexed: [{'id': '181a64c3-2210-4944-b9a8-881f455595f7', 'text': 'he average air temperature in 2023 is 1The average air temperature in 2023 is $1 1 . 5 ^ { \\circ } \\mathrm { C }$ nd the average river temperature is and the average river temperature is $1 3 . 1 ^ { \\circ } \\mathrm { C }$ The river temperature follows the air temperature except slightly shifted and with . The river temperature follows the air temperature except slightly shifted and with flattened peaks. On the basis of these data, the losses of short-term TES, the COP of the river water HP, or the solar yield of the ST plant are affected.', 'heading': '2.3. Application Case and Assumptions'}]
